# Capstone 4 Colab Training Runbook
This notebook executes real QLoRA training using the project codebase and saves artifacts to Google Drive.

## 1) Runtime Setup
Use Runtime > Change runtime type > GPU (T4 recommended).

In [26]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2) Project Path
Set this path to your project directory in Google Drive.

In [28]:
import os
PROJECT_DIR = '/content/drive/MyDrive/capstone1/project'
assert os.path.isdir(PROJECT_DIR), f'Project folder not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())

Working dir: /content/drive/MyDrive/capstone1/project


In [ ]:
# ── Dependency & triton compatibility setup ─────────────────────────────────
# triton 3.x removed the ops submodule; bitsandbytes still imports from it.
# Stub it out FIRST so subsequent pip installs and imports don't crash.
import sys, types
if 'triton.ops' not in sys.modules:
    try:
        import triton
        _stub = types.ModuleType('triton.ops')
        sys.modules['triton.ops'] = _stub
        if not hasattr(triton, 'ops'):
            triton.ops = _stub
    except ImportError:
        pass  # triton not installed yet, will be handled after pip install

!python -m pip uninstall -y torchao -q
!python -m pip install -q --upgrade pip
!python -m pip install -q 'bitsandbytes>=0.42.0,<0.44.0'
!python -m pip install -q -r requirements.txt

# Re-apply stub after install in case triton was just installed.
if 'triton.ops' not in sys.modules:
    try:
        import triton
        _stub = types.ModuleType('triton.ops')
        sys.modules['triton.ops'] = _stub
        if not hasattr(triton, 'ops'):
            triton.ops = _stub
    except ImportError:
        pass
print('Dependency setup complete. triton.ops stub active:', 'triton.ops' in sys.modules)

## 3) Environment Variables
Set Phi-3.5 model and local/cache loading options.

In [30]:
os.environ['MODEL_BASE_ID'] = 'microsoft/Phi-3.5-mini-instruct'
os.environ['MODEL_BASE_PATH'] = '/content/drive/MyDrive/capstone1/Phi-3.5-mini-instruct'  # optional local folder
os.environ['USE_HF_BACKEND'] = '1'
os.environ['HF_LOCAL_FILES_ONLY'] = '1'  # use local/cache only; set to 0 if download is allowed
os.environ['MAX_NEW_TOKENS'] = '384'

# Optional token only when HF_LOCAL_FILES_ONLY=0 and download/auth is required.
# os.environ['HF_TOKEN'] = 'hf_xxx'

if os.environ['HF_LOCAL_FILES_ONLY'] == '1':
    # If MODEL_BASE_PATH does not exist, transformers will fall back to cache for MODEL_BASE_ID.
    if os.environ.get('MODEL_BASE_PATH') and not os.path.isdir(os.environ['MODEL_BASE_PATH']):
        print('MODEL_BASE_PATH not found, loader will use local HF cache with MODEL_BASE_ID.')

## 4) Validate Dataset
Ensure train/eval files exist and pass schema checks.

In [31]:
!python -m src.data.validate_dataset --train data/raw/train.jsonl --eval data/raw/eval.jsonl --report artifacts/reports/dataset_validation_report.json

Train rows: 160
Eval rows: 40
Total issues: 0
Report written: artifacts/reports/dataset_validation_report.json


## 5) Start Real QLoRA Training
This runs full training and auto-promotes the adapter to artifacts/models/latest.

In [32]:
# Preflight check: verify MODEL_BASE_PATH and local HF cache availability
import os
from pathlib import Path

model_id = os.environ.get('MODEL_BASE_ID', 'microsoft/Phi-3.5-mini-instruct').strip()
model_base_path = os.environ.get('MODEL_BASE_PATH', '').strip()
local_only = os.environ.get('HF_LOCAL_FILES_ONLY', '0').strip()
print('=== Preflight: Model Availability ===')
print('MODEL_BASE_ID:', model_id)
print('MODEL_BASE_PATH:', model_base_path if model_base_path else '<not set>')
print('HF_LOCAL_FILES_ONLY:', local_only)

path_ok = False
if model_base_path:
    p = Path(model_base_path)
    path_ok = p.exists() and p.is_dir()
    print(f'MODEL_BASE_PATH exists: {path_ok}')
    if path_ok:
        has_model_files = any((p / name).exists() for name in ['config.json', 'tokenizer.json', 'tokenizer_config.json'])
        print(f'MODEL_BASE_PATH looks like a model folder: {has_model_files}')

cache_roots = [
    Path('/root/.cache/huggingface/hub'),
    Path('/content/.cache/huggingface/hub'),
    Path(os.environ.get('HF_HOME', '')).expanduser() / 'hub' if os.environ.get('HF_HOME') else None,
]
cache_roots = [c for c in cache_roots if c is not None]

needle = model_id.replace('/', '--').lower()
cache_hits = []
for root in cache_roots:
    if root.exists():
        for d in root.glob('models--*'):
            if needle in d.name.lower():
                cache_hits.append(str(d))

print('HF cache locations checked:')
for root in cache_roots:
    print(' -', root)
print('Local cache has model:', len(cache_hits) > 0)
if cache_hits:
    print('Cache match examples:')
    for hit in cache_hits[:3]:
        print('   ', hit)

if local_only == '1' and not path_ok and not cache_hits:
    raise RuntimeError(
        'HF_LOCAL_FILES_ONLY=1 but model not found in MODEL_BASE_PATH or local cache. '
        'Set MODEL_BASE_PATH correctly or disable local-only mode for initial download.'
    )

print('Preflight check passed.')

=== Preflight: Model Availability ===
MODEL_BASE_ID: microsoft/Phi-3.5-mini-instruct
MODEL_BASE_PATH: /content/drive/MyDrive/capstone1/Phi-3.5-mini-instruct
HF_LOCAL_FILES_ONLY: 1
MODEL_BASE_PATH exists: True
MODEL_BASE_PATH looks like a model folder: True
HF cache locations checked:
 - /root/.cache/huggingface/hub
 - /content/.cache/huggingface/hub
Local cache has model: False
Preflight check passed.


In [33]:
!python -m src.training.run_qlora --execute-training --notes 'colab real run'

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 195/195 [00:38<00:00,  5.11it/s, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the disk and cpu.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/drive/MyDrive/capstone1/project/src/training/run_qlora.py", line 206, in <module>
    main()
  File "/content/drive/MyDrive/capstone1/project/src/training/run_qlora.py", line 186, in main
    metrics = _run

## 6) Verify Backend and Adapter Load
Checks base and adapter-backed backends and writes verification report.

In [21]:
!python -m src.inference.verify_backend --run-generate --report artifacts/reports/backend_verification.json

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 195/195 [05:53<00:00,  1.81s/it, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the cpu and disk.
Loading weights:   6% 11/195 [00:03<00:40,  4.58it/s, Materializing param=model.layers.1.mlp.gate_up_proj.weight]       ^C


## 7) Run Full 4-Variant Eval
Produces the final scorecard report over eval.jsonl.

In [22]:
!python -m src.eval.run_eval --eval data/raw/eval.jsonl --report artifacts/reports/variant_eval_report_full.json

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 195/195 [00:36<00:00,  5.39it/s, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the disk and cpu.
Loading weights:   9% 17/195 [00:04<00:43,  4.06it/s, Materializing param=model.layers.2.mlp.gate_up_proj.weight]        ^C


## 8) Run Golden-Only 4-Variant Eval
Runs evaluation on the fixed golden id subset for stable trend comparisons.

In [ ]:
!python -m src.eval.run_eval --eval data/raw/eval.jsonl --golden-ids data/golden/golden_eval_ids.json --report artifacts/reports/variant_eval_report_golden.json

## 9) Notebook 4-Variant Report View
Load the eval report generated above and inspect metrics/charts directly in the notebook.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

report_path = Path('artifacts/reports/variant_eval_report_full.json')
assert report_path.exists(), f'Report not found: {report_path}'

report = json.loads(report_path.read_text(encoding='utf-8'))
records = report.get('records', [])
assert records, 'No records found in eval report.'

# Flatten per-record variant outputs into a tabular dataframe.
rows = []
for rec in records:
    rec_id = rec.get('id')
    for v in rec.get('variant_outputs', []):
        rows.append({
            'id': rec_id,
            'variant': v.get('variant'),
            'latency_ms': float(v.get('latency_ms', 0.0)),
            'token_estimate': float(v.get('token_estimate', 0.0)),
            'format_adherence': float(v.get('format_adherence', 0.0)),
            'category_match': float(v.get('category_match', 0.0)),
            'severity_match': float(v.get('severity_match', 0.0)),
            'lexical_overlap': float(v.get('lexical_overlap', 0.0)),
        })

df = pd.DataFrame(rows)
display(df.head(10))

# Aggregate by variant for quick scorecard view.
agg = (
    df.groupby('variant', as_index=False)
    .agg(
        format_adherence=('format_adherence', 'mean'),
        latency_ms=('latency_ms', 'mean'),
        token_estimate=('token_estimate', 'mean'),
        category_match=('category_match', 'mean'),
        severity_match=('severity_match', 'mean'),
        lexical_overlap=('lexical_overlap', 'mean'),
    )
)
display(agg)

fig1 = px.bar(agg, x='variant', y='format_adherence', title='Avg Format Adherence by Variant', range_y=[0, 1])
fig2 = px.bar(agg, x='variant', y='latency_ms', title='Avg Latency by Variant (ms)')
fig3 = px.bar(agg, x='variant', y='token_estimate', title='Avg Token Estimate by Variant')
fig4 = px.bar(
    agg,
    x='variant',
    y=['category_match', 'severity_match', 'lexical_overlap'],
    barmode='group',
    title='Avg Correctness Proxy Metrics',
)

fig1.show()
fig2.show()
fig3.show()
fig4.show()

## 10) Notebook 4-Variant Single Prompt Compare
Run all four variants for one prompt directly in notebook (equivalent to compare app behavior).

In [ ]:
import json
from pathlib import Path

from src.config.settings import load_settings
from src.inference.adapter_registry import resolve_latest_adapter
from src.inference.router import InferenceRouter

settings = load_settings()
adapter_path = resolve_latest_adapter(settings.models_dir) or settings.adapter_path
print('Adapter path:', adapter_path)

eval_file = Path('data/raw/eval.jsonl')
eval_rows = [json.loads(line) for line in eval_file.read_text(encoding='utf-8').splitlines() if line.strip()]

row_idx = 0  # change row index as needed
row = eval_rows[row_idx]

prompt = (
    f"{row.get('task_instruction', '').strip()}\n\n"
    f"File: {row.get('input', {}).get('file_path', 'unknown')}:{row.get('input', {}).get('line_hint', '')}\n"
    f"Context: {row.get('input', {}).get('context_summary', '')}\n\n"
    f"Changed code:\n{row.get('input', {}).get('changed_code', '')}\n"
)

router = InferenceRouter(adapter_path=adapter_path)
results = router.run_all_with_timings(prompt)

for result in results:
    print('=' * 80)
    print('VARIANT:', result.get('variant'))
    print('LATENCY_MS:', result.get('latency_ms'))
    if result.get('metadata'):
        print('METADATA:', result.get('metadata'))
    print('-' * 80)
    print(result.get('text', ''))

## 11) Launch Comparison UI (Optional in Colab)
For local usage, run Streamlit in your workstation. In Colab, this command can still be used with tunneling if needed.

In [ ]:
# !streamlit run src/app/compare_app.py